# ABSA Experiment
Experiment to make extract keywords from the task 1 and passing the keywords result to task 3

Author: Raihan Muhammad Daffa<br/>
NIM: 18223140

## Import Modules

In [1]:
import os
import sys
import random

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import numpy as np
import re
import json
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, set_seed


SEED = 42

def reproducible(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
reproducible(SEED)

## GPU Utilization Check

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## Helper

In [3]:
def build_prompt(review_text):
    return(
        "### Instruction: Extract the key aspects and their sentiments from the following game review"
        "You MUST format the output strictly as Aspects: "
        "[category:aspect:sentiment, ...]\n\n"
        f"### Input: Review: \"{review_text}\"\n\n"
        "### Response:"
    )
    
def parse_aspect_tuples(text):
    clean_text = re.sub(r"Aspects\s*:\s*\[?", "", text, flags=re.IGNORECASE).replace("]", "")
    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", clean_text)
    return set((cat.strip().lower(), asp.strip().lower(), sent.strip().lower()) for cat, asp, sent in matches)

def evaluate_model(model, tokenizer, test_data, experiment_name):
    model.eval()
    exact_matches = 0
    total_p, total_r, total_f1 = 0.0, 0.0, 0.0
    n = len(test_data)
    
    for item in test_data:
        inputs = tokenizer(item["input_text"], return_tensors="pt", max_length=1024, truncation=True).to(device)
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                repetition_penalty=1.0,
                length_penalty=1.5,
                early_stopping=True
            )
            
        pred_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        gold_text = item["target_text"]
        
        pred_triplets = parse_aspect_tuples(pred_text)
        gold_triplets = parse_aspect_tuples(gold_text)
        
        if pred_triplets == gold_triplets and len(gold_triplets) > 0:
            exact_matches += 1

        intersection = len(pred_triplets & gold_triplets)
        
        if pred_triplets:
            prec = intersection / len(pred_triplets)
        else:
            prec = 0.0
            
        if gold_triplets:
            rec = intersection / len(gold_triplets)
        else:  
            rec = 0.0
        
        if prec + rec > 0:
            f1 = 2 * (prec * rec) / (prec + rec)
        else:
            f1 = 0.0
        
        total_p += prec
        total_r += rec
        total_f1 += f1
        
    results = {
        "Experiment": experiment_name,
        "Exact Match (%)": round((exact_matches / n) * 100, 2),
        "Precision (%)": round((total_p / n) * 100, 2),
        "Recall (%)": round((total_r / n) * 100, 2),
        "F1-Score (%)": round((total_f1 / n) * 100, 2)
    }
    return results

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    preds = np.where(preds != -100, preds, bart_tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, bart_tokenizer.pad_token_id)
    decoded_preds = bart_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)
    exact_matches = 0
    
    total_p, total_r, total_f1 = 0.0, 0.0, 0.0
    n = len(decoded_labels)
    
    for pred_text, gold_text in zip(decoded_preds, decoded_labels):
        pred_triplets = parse_aspect_tuples(pred_text)
        gold_triplets = parse_aspect_tuples(gold_text)
        if pred_triplets == gold_triplets and len(gold_triplets) > 0:
            exact_matches += 1
        intersection = len(pred_triplets & gold_triplets)
        prec = intersection / len(pred_triplets) if pred_triplets else 0.0
        rec = intersection / len(gold_triplets) if gold_triplets else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        total_p += prec
        total_r += rec
        total_f1 += f1
        
    return {
        "em": round((exact_matches / n) * 100, 2),
        "precision": round((total_p / n) * 100, 2),
        "recall": round((total_r / n) * 100, 2),
        "f1": round((total_f1 / n) * 100, 2),
    }

all_expe_results = []

In [4]:
class ABSADataset(Dataset):
    def __init__(self, data_pairs, tokenizer, max_input_len=1024, max_target_len=128):
        self.data_pairs = data_pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        
    def __len__(self):
        return len(self.data_pairs)
    
    def __getitem__(self, idx):
        item = self.data_pairs[idx]
        model_inputs = self.tokenizer(item["input_text"], max_length=self.max_input_len, truncation=True)
        labels = self.tokenizer(text_target=item["target_text"], max_length=self.max_target_len, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

## Load the dataset

In [5]:
INPUT_FILE = os.path.join(BASE_DIR, "data", "preprocessed", "relevant_reviews_labeled.jsonl")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

data_pairs = []
for game_id, game_data in raw_data.items():
    for review_pair in game_data.get("review_pairs", []):
        if review_pair["target_label"] != "Aspects: []":
            data_pairs.append({
                "game": game_data.get("app_name", f"Game {game_id}"),
                "input_text": build_prompt(review_pair["review"]),
                "target_text": review_pair["target_label"]
            })

## Split the data

In [6]:
n_data = len(data_pairs)
n_train = int(0.70 * n_data)
n_val = int(0.85 * n_data)

train_data = data_pairs[:n_train]
val_data = data_pairs[n_train:n_val]
test_data = data_pairs[n_val:]

print(f"Total samples: {n_data} | Train: {len(train_data)} |  Validation: {len(val_data)} | Test: {len(test_data)}")

Total samples: 1589 | Train: 1112 |  Validation: 238 | Test: 239


## Experiment 1 (Zero-Base Shot)

In [7]:
base_model_name = "facebook/bart-base"
bart_tokenizer = BartTokenizer.from_pretrained(base_model_name)
bart_zero_shot_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [8]:
res_zero_shot = evaluate_model(bart_zero_shot_model, bart_tokenizer, test_data, "Zero-Shot (Raw BART)")
all_expe_results.append(res_zero_shot)
print(pd.DataFrame([res_zero_shot]))

             Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Zero-Shot (Raw BART)              0.0            0.0         0.0   

   F1-Score (%)  
0           0.0  


In [9]:
del bart_zero_shot_model
torch.cuda.empty_cache()

## Experiment 2 (Task-Adapted BART)

In [10]:
bart_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)
bart_train_dataset = ABSADataset(train_data, bart_tokenizer)
bart_val_dataset = ABSADataset(val_data, bart_tokenizer)
bart_data_collator = DataCollatorForSeq2Seq(tokenizer=bart_tokenizer,model=bart_task_adapted_model)

bart_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_4epochs",
    num_train_epochs=4,
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=2, 
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=10,
    logging_strategy="epoch", 
    eval_strategy="epoch",
    generation_num_beams=2,  
    lr_scheduler_type="cosine",
    save_strategy="no",
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

bart_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_task_adapted_model,
    args=bart_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    eval_dataset=bart_val_dataset,
    data_collator=bart_data_collator,
    compute_metrics=compute_metrics 
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [11]:
bart_task_adapted_trainer.train()

Epoch,Training Loss,Validation Loss,Em,Precision,Recall,F1
1,1.199418,0.180291,35.290000,54.830000,48.320000,49.930000
2,0.373404,0.151328,45.380000,62.390000,57.560000,58.610000
3,0.261562,0.141328,44.120000,62.180000,60.080000,59.660000
4,0.181068,0.149748,42.020000,60.500000,57.560000,57.560000


d:\Data\Kuliah\Tingkat 2\Others\Coding\DatSci\IF5153_NLP_Final_Projects\.venv\Lib\site-packages\transformers\generation\utils.py:1977: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrainOutput(global_step=1112, training_loss=0.5038627205992774, metrics={'train_runtime': 272.3731, 'train_samples_per_second': 16.331, 'train_steps_per_second': 4.083, 'total_flos': 789064475996160.0, 'train_loss': 0.5038627205992774, 'epoch': 4.0})

In [12]:
bart_task_adapted_evaluation = evaluate_model(bart_task_adapted_model, bart_tokenizer, test_data, "Task-Adapted (BART - 4 Epochs)")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [13]:
all_expe_results.append(bart_task_adapted_evaluation)
print(pd.DataFrame([bart_task_adapted_evaluation]))

                       Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Task-Adapted (BART - 4 Epochs)             47.7           59.0       60.81   

   F1-Score (%)  
0         59.09  


## Experiments 3

In [14]:
bart_hyperparametered_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)

bart_hyperparametered_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_6epochs",
    num_train_epochs=6,
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=2, 
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    lr_scheduler_type="cosine",
    save_strategy="no",
    logging_strategy="epoch", 
    eval_strategy="epoch",
    generation_num_beams=2,
    predict_with_generate=True,
    fp16=True,    
    report_to="none"
)

bart_hyperparametered_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_hyperparametered_task_adapted_model,
    args=bart_hyperparametered_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    eval_dataset=bart_val_dataset,
    data_collator=bart_data_collator,
    compute_metrics=compute_metrics 
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [15]:
bart_hyperparametered_task_adapted_trainer.train()

Epoch,Training Loss,Validation Loss,Em,Precision,Recall,F1
1,4.447782,0.205143,32.770000,51.050000,43.910000,45.940000
2,0.895857,0.187003,32.770000,52.100000,42.440000,45.660000
3,0.671805,0.133062,39.920000,59.450000,56.300000,56.230000
4,0.473977,0.137982,43.700000,62.820000,59.450000,59.520000
5,0.343692,0.138135,42.860000,61.340000,57.350000,57.840000
6,0.279733,0.139055,44.960000,63.450000,59.450000,59.940000


d:\Data\Kuliah\Tingkat 2\Others\Coding\DatSci\IF5153_NLP_Final_Projects\.venv\Lib\site-packages\transformers\generation\utils.py:1977: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrainOutput(global_step=834, training_loss=1.185474258532627, metrics={'train_runtime': 1794.4183, 'train_samples_per_second': 3.718, 'train_steps_per_second': 0.465, 'total_flos': 1184659584675840.0, 'train_loss': 1.185474258532627, 'epoch': 6.0})

In [16]:
bart_hyperparametered_task_adapted_evaluation = evaluate_model(bart_hyperparametered_task_adapted_model, bart_tokenizer, test_data, "Hyperparametered Task-Adapted (BART - 6 Epochs)")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [17]:
all_expe_results.append(bart_hyperparametered_task_adapted_evaluation)
print(pd.DataFrame([bart_hyperparametered_task_adapted_evaluation]))

                                        Experiment  Exact Match (%)  \
0  Hyperparametered Task-Adapted (BART - 6 Epochs)            48.54   

   Precision (%)  Recall (%)  F1-Score (%)  
0          60.04       61.65          60.0  


## Evaluation

In [18]:
summary_df = pd.DataFrame(all_expe_results)
display(summary_df)

,Experiment,Exact Match (%),Precision (%),Recall (%),F1-Score (%)
0,Zero-Shot (Raw BART),0.00,0.00,0.00,0.00
1,Task-Adapted (BART - 4 Epochs),47.70,59.00,60.81,59.09
2,Hyperparametered Task-Adapted (BART - 6 Epochs),48.54,60.04,61.65,60.00


## Error Analysis

In [19]:
best_model = bart_task_adapted_model
best_model.eval()

for sample in test_data[:8]:
    inputs = bart_tokenizer(sample["input_text"], return_tensors="pt", max_length=1024, truncation=True).to(device)
    
    with torch.no_grad():
        out_ids = best_model.generate(
            **inputs, 
            max_new_tokens=128, 
            num_beams=4,
            length_penalty=1.5, 
            repetition_penalty=1.0,
            early_stopping=True
            )
    
    pred = bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    print(f"\n Game: {sample['game']}")
    print(f"Ground Truth: {sample['target_text']}")
    print(f"Prediction: {pred}")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Bastion
Ground Truth: Aspects: [Gameplay:gameplay:positive, Gameplay:story:positive]
Prediction: Aspects: [Gameplay:gameplay:positive, Gameplay:story:positive]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Bastion
Ground Truth: Aspects: [Gameplay:gameplay:positive]
Prediction: Aspects: [Gameplay:gameplay:positive, Gameplay:story:positive]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Bastion
Ground Truth: Aspects: [Audio:music:positive]
Prediction: Aspects: [Audio:music:positive]


... (2 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Resident Evil Revelations 2
Ground Truth: Aspects: [Gameplay:campaign:positive]
Prediction: Aspects: [Gameplay:campaign:positive]

 Game: Resident Evil Revelations 2
Ground Truth: Aspects: [Gameplay:graphics:negative, Gameplay:multiplayer:negative]
Prediction: Aspects: [Gameplay:graphics:positive, Gameplay:multiplayer:positive]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Lords Of The Fallen
Ground Truth: Aspects: [Gameplay:combat:negative]
Prediction: Aspects: [Gameplay:combat:negative, UI/UX:controls:negative]

 Game: Lords Of The Fallen
Ground Truth: Aspects: [Gameplay:multiplayer:negative]
Prediction: Aspects: [Gameplay:multiplayer:positive]


## Export the JSON File

In [20]:
def parse_predicted_aspects(pred_text):
    cleaned = re.sub(r"Aspects\s*:\s*\[?", "", pred_text, flags=re.IGNORECASE)
    cleaned = cleaned.replace("]", "")
    
    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", cleaned)
    aspects_list = []
    for cat, asp, sent in matches:
        aspects_list.append({
            "category": cat.strip(),
            "aspect": asp.strip(),
            "sentiment": sent.strip().capitalize()
        })
    return aspects_list

In [ ]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    games_data = json.load(f)
    
pred_output = {}
best_model.eval()

for game_id, data in games_data.items():
    reviews = data.get("reviews", [])
    all_aspects = []
    seen = set()
    raw_preds = []
    
    for review in reviews:
        prompt = build_prompt(review)
        inputs = bart_tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to(device)
        with torch.no_grad():
            out_ids = best_model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                repetition_penalty=1.0,
                length_penalty=1.5,
                early_stopping=True
            )
            
        pred_text = bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)
        raw_preds.append(pred_text)
        for item in parse_predicted_aspects(pred_text):
            if item["aspect"].lower() not in review.lower():
                continue
            
            key = (item["category"].lower(), item["aspect"].lower())
            if key not in seen:
                seen.add(key)
                all_aspects.append(item)
                
    pred_output[game_id] = {
        "app_name": data.get("app_name", ""),
        "reviews": reviews,
        "aspects": all_aspects,
        "raw_prediction": raw_preds
    }

In [ ]:
pred_output_file = os.path.join(BASE_DIR, "data", "modeled" , "keywords_predicted.jsonl")
os.makedirs(os.path.dirname(pred_output_file), exist_ok=True)
with open(pred_output_file, "w", encoding="utf-8") as f:
    json.dump(pred_output, f, indent=4, ensure_ascii=False)